# Synthetic Data Generation Component Simulation Framework


This notebook runs the **Synthetic Data Generation Component** (Agentic Simulation Cycle) — a simple loop-based simulator that models multi-actor deliberation on a given topic.


## How it works


The simulation uses a **time-step loop** with simple actor checks:


1. **Actors** are initialized with behavioral parameters that control how often they post, reply, and vote.
2. Each loop step checks whether an actor should create a **POST** or an **ACTION** based on its configured step intervals.
3. Smaller step intervals make an actor act more often. Larger step intervals slow that actor down.
4. A **Bernoulli gate** (`theta_action`) filters votes — an actor only casts a vote when `U(0,1) > theta_action`, so higher values make voting rarer.
5. The loop runs until the time horizon `T` (in minutes) is exceeded.

## Setup

Make sure you have the required packages installed:

```
pip install openai numpy python-dotenv
```

You also need an `OPENAI_API_KEY` — either in a `.env` file at the project root or set as an environment variable.

In [ ]:
import os
import sys
import json
import asyncio
import logging

from dotenv import load_dotenv

# Add the project root to the path so we can import the src package
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

load_dotenv(os.path.join(PROJECT_ROOT, ".env"))

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(name)s - %(message)s",
)
logging.getLogger("httpx").setLevel(logging.WARNING)

In [ ]:
from src.llm_client import OpenAIClient
from src.actors import build_actor_roster
from src.simulation import run_simulation
from src.config import (
    TOPIC_TITLE,
    TOPIC_SUMMARY,
    SIMULATION_HORIZON_MINUTES,
    ACTOR_PARAMS,
)

## Configuration

You can override the defaults below before running the simulation.

In [ ]:
# ── Topic ──────────────────────────────────────────────────────────────────
# Change these to simulate a different deliberation topic.

topic_title = TOPIC_TITLE
topic_summary = TOPIC_SUMMARY

print(f"Topic:   {topic_title}")
print(f"Summary: {topic_summary[:100]}...")

In [ ]:
# T is in minutes. Longer horizons produce more events but cost more API calls.

T = SIMULATION_HORIZON_MINUTES  # default: 20.0
print(f"Horizon: {T} minutes")

In [ ]:
# Review the default actor parameters from config.

import pandas as pd

pd.DataFrame(ACTOR_PARAMS)

## Run the Simulation

In [ ]:
llm_client = OpenAIClient(api_key=os.environ["OPENAI_API_KEY"])
actors = build_actor_roster()

print(f"Actors: {len(actors)}")
for a in actors:
    print(
        f"  - {a.name} ({a.archetype})  "
        f"post_every_n_steps={a.post_every_n_steps}  "
        f"action_every_n_steps={a.action_every_n_steps}  "
        f"p_reply={a.p_reply}  theta_action={a.theta_action}"
    )

In [ ]:
import time
import random

# Keep notebook output readable during long runs
logging.getLogger("src.simulation").setLevel(logging.WARNING)

MAX_WALL_CLOCK_SECONDS = 120
ENABLE_LOCAL_FALLBACK_ON_API_ERROR = True

wall_clock_start = time.time()
H = None

try:
    H = await asyncio.wait_for(
        run_simulation(
            actors=actors,
            llm_client=llm_client,
            topic_title=topic_title,
            topic_summary=topic_summary,
            T=T,
        ),
        timeout=MAX_WALL_CLOCK_SECONDS,
    )
except asyncio.TimeoutError:
    elapsed = time.time() - wall_clock_start
    print(f"[TIMEOUT] Simulation exceeded {MAX_WALL_CLOCK_SECONDS}s and stopped safely.")
    print(f"  Elapsed wall-clock: {elapsed:.1f}s")
except Exception as e:
    elapsed = time.time() - wall_clock_start
    print(f"[ERROR] Simulation stopped: {type(e).__name__}: {e}")
    print(f"  Elapsed wall-clock: {elapsed:.1f}s")

if H is None and ENABLE_LOCAL_FALLBACK_ON_API_ERROR:
    from src.models import SharedHistory

    print("[Fallback] Building local synthetic history so analysis/export can continue.")
    H = SharedHistory()

    for idx, actor in enumerate(actors[: min(len(actors), 12)]):
        reply_to = None
        if idx > 0 and idx % 3 == 0 and H.posts:
            reply_to = random.choice(H.posts)["id"]

        text = (
            f"[{actor.name}] Local fallback comment for '{topic_title}'. "
            "Generated without external API."
        )

        H.add_post(
            post_id=f"local-{idx+1}",
            t=round(min(T, 0.1 * (idx + 1)), 3),
            actor=actor.name,
            archetype=actor.archetype,
            text=text,
            age=random.randint(25, 70),
            nationality=random.choice(["Greece", "Cyprus", "Bulgaria", "Romania", "Albania"]),
            political_background=random.choice(["Moderate", "Liberal", "Conservative"]),
            fan_type=random.choice(["Casual", "Die-hard", "Fair-weather"]),
            reply_to=reply_to,
        )

elapsed = time.time() - wall_clock_start

if H is not None:
    print(f"\nSimulation complete: {len(H)} total events")
    print(f"  Posts: {len(H.posts)}")
    print(f"  Votes: {len(H.votes)}")
    print(f"  Wall-clock: {elapsed:.1f}s")
else:
    print("\nSimulation did not produce history.")

## Explore Results

In [ ]:
# Convert to a DataFrame for easier analysis
events = H.all_events() if H is not None else []
events_df = pd.DataFrame(events)

if events_df.empty:
    print("No events available for analysis.")
else:
    events_df.head(10)

In [ ]:
# Posts per actor
if events_df.empty:
    print("No post events available.")
    posts_df = pd.DataFrame()
else:
    posts_df = events_df[events_df["type"] == "post"]
    if posts_df.empty:
        print("No post events available.")
    else:
        posts_df["actor"].value_counts()

In [ ]:
# Votes per actor
if events_df.empty:
    print("No vote events available.")
    votes_df = pd.DataFrame()
else:
    votes_df = events_df[events_df["type"] == "vote"]
    if votes_df.empty:
        print("No vote events available.")
    else:
        votes_df["actor"].value_counts()

In [ ]:
# Vote type distribution
if "votes_df" not in globals() or votes_df.empty:
    print("No vote distribution to show.")
else:
    print(votes_df["vote"].value_counts())

In [ ]:
# Reply vs. top-level post breakdown
if "posts_df" not in globals() or posts_df.empty:
    print("No posts available for reply breakdown.")
else:
    replies = posts_df["reply_to"].notna().sum()
    top_level = posts_df["reply_to"].isna().sum()
    print(f"Top-level comments: {top_level}")
    print(f"Replies:            {replies}")

In [ ]:
# Timeline: events over simulated time
if events_df.empty:
    print("No timeline available.")
else:
    # Text-based histogram to avoid optional plotting dependency.
    bin_count = 5
    binned = pd.cut(events_df["t"], bins=bin_count)
    print("Event distribution over time bins:")
    print(binned.value_counts().sort_index())

## Save Results

In [ ]:
output_dir = os.path.join(PROJECT_ROOT, "session")
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, "simulation_output.json")

if H is None or len(H) == 0:
    print("No events to save. Skipping simulation_output.json write.")
else:
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(H.all_events(), f, indent=2, ensure_ascii=False)

    print(f"Saved {len(H)} events to {output_path}")

In [ ]:
import os
from minio import Minio
from minio.error import S3Error

class MinIOClient:
    def __init__(self, endpoint: str = None, access_key: str = None, secret_key: str = None, secure: bool = None):
        """
        Αρχικοποιεί τον MinIO Client. 
        Αν δεν δοθούν ορίσματα, διαβάζει αυτόματα από τα Environment Variables (.env).
        """
        self.endpoint = endpoint or os.getenv("MINIO_ENDPOINT", "localhost:9000")
        self.access_key = access_key or os.getenv("MINIO_ACCESS_KEY", "minioadmin")
        self.secret_key = secret_key or os.getenv("MINIO_SECRET_KEY", "minioadmin")
        
        if secure is None:
            # Μετατροπή του string "False"/"True" από το .env σε boolean
            secure_str = os.getenv("MINIO_SECURE", "False").lower()
            self.secure = secure_str in ("true", "1")
        else:
            self.secure = secure

        # Δημιουργία του εσωτερικού Minio instance
        self.client = Minio(
            self.endpoint,
            access_key=self.access_key,
            secret_key=self.secret_key,
            secure=self.secure
        )

    def upload_file(self, file_path: str, bucket_name: str, object_name: str = None) -> bool:
        """
        Ανεβάζει ένα αρχείο στο MinIO bucket. Αν το bucket δεν υπάρχει, το δημιουργεί.
        """
        try:
            if object_name is None:
                object_name = os.path.basename(file_path)
            
            if not os.path.exists(file_path):
                print(f"File not found: {file_path}")
                return False
            
            # Έλεγχος και αυτόματη δημιουργία bucket αν λείπει
            if not self.client.bucket_exists(bucket_name):
                self.client.make_bucket(bucket_name)
                print(f"Created missing bucket: '{bucket_name}'")
            
            file_size = os.path.getsize(file_path)
            
            with open(file_path, 'rb') as file_data:
                self.client.put_object(
                    bucket_name,
                    object_name,
                    file_data,
                    length=file_size
                )
            
            return True
        
        except S3Error as e:
            print(f"MinIO error: {e}")
            return False
        except Exception as e:
            print(f"Error uploading file: {e}")
            return False
        


In [ ]:
from src.minio_manager import MinIOClient

# Ρύθμιση παραμέτρων
bucket_name = "certain-results"
object_name = "simulation_output.json"

if not os.path.exists(output_path) or os.path.getsize(output_path) == 0:
    print("Upload skipped: simulation_output.json is missing or empty.")
else:
    # Αρχικοποίηση (τραβάει αυτόματα τα keys από το load_dotenv που έτρεξες στην αρχή)
    minio_client = MinIOClient()

    # Ανέβασμα του αρχείου
    success = minio_client.upload_file(output_path, bucket_name, object_name)

    if success:
        print(f"Uploaded {object_name} to MinIO bucket '{bucket_name}'")
    else:
        print("Upload failed.")

## Export Results with Metadata

In [ ]:
from src.result_exporter import SimulationResultExporter

result_json = None

# Εξαγωγή αποτελεσμάτων στο σχήμα που περιέχει Metadata
print("[Exporting results with metadata...]")

if H is None or len(H.posts) == 0:
    print("[WARNING] No posts available. Skipping metadata export.")
else:
    result = SimulationResultExporter.export_as_dataset(
        H,
        topic_title=topic_title,
        topic_summary=topic_summary,
        include_metadata=True,
    )

    result_json = SimulationResultExporter.to_json_string(result)
    print(f"[OK] Result exported ({len(result_json)} bytes)")

    # Αποθήκευση τοπικά
    output_path_metadata = os.path.join(output_dir, "simulation_output_with_metadata.json")
    with open(output_path_metadata, "w", encoding="utf-8") as f:
        f.write(result_json)

    print(f"[OK] Saved locally: {output_path_metadata}")

    # Ανέβασμα στο Minio
    try:
        minio_client_meta = MinIOClient()
        bucket_name_meta = "certain-results"
        object_name_meta = "simulation_output_with_metadata.json"

        success_meta = minio_client_meta.upload_file(output_path_metadata, bucket_name_meta, object_name_meta)

        if success_meta:
            print(f"[OK] Uploaded {object_name_meta} to MinIO bucket '{bucket_name_meta}'")
        else:
            print("[WARNING] Failed to upload metadata file to MinIO")

    except Exception as e:
        print(f"[WARNING] Minio upload failed: {e}")

In [ ]:
import json

if not result_json:
    print("[Output Verification] Skipped (no metadata export output).")
else:
    # Επαλήθευση του output
    parsed_result = json.loads(result_json)

    print("[Output Verification]")
    print(f"  Topic: {parsed_result.get('TopicTitle')}")
    print(f"  Total comments: {len(parsed_result.get('Dataset', []))}")

    # Μέτρηση με/χωρίς metadata
    comments_with_metadata = sum(
        1
        for r in parsed_result.get("Dataset", [])
        if r.get("Metadata") and r["Metadata"].get("FirstName") is not None
    )
    comments_without_metadata = sum(
        1
        for r in parsed_result.get("Dataset", [])
        if r.get("Metadata") is None or r["Metadata"].get("FirstName") is None
    )

    print("\n[Comments]")
    print(f"  With metadata (identified): {comments_with_metadata}")
    print(f"  Without metadata (anonymous): {comments_without_metadata}")

    # Δείγμα πρώτων σχολίων
    print("\n[Sample Comments]")
    for i, comment in enumerate(parsed_result.get("Dataset", [])[:3]):
        metadata = comment.get("Metadata")
        if metadata and metadata.get("FirstName"):
            author = f"{metadata.get('FirstName')} {metadata.get('LastName', '')}"
        else:
            author = "[Anonymous]"

        text = comment.get("Text", "")[:50]
        print(f"  {i+1}. {author:20s}: {text}...")